# Directed question classifier: how the models are doing

Every question gets one of three labels:
- **General**: anyone can answer it, so it stays.
- **Fixable**: it's aimed at a group, but rewording it fixes that.
- **Unsalvageable**: only that group can answer it.

The **true class** (what we're trying to catch) is **directed**, meaning fixable or unsalvageable. General is the negative class. The old rules only say yes or no, so they can only be scored on directed vs general.

| Contender | What it is |
|---|---|
| Dummy | Always says general. A model has to beat this to show it's learning anything. |
| Rules | The old regex rules. A model has to beat these to be worth replacing them. |
| Naive Bayes | The first real model. |

In [20]:
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root))

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix

from src.ml.directed.common import TRAIN_PATH, LABELS
from src.ml.directed.score_rules import score_rules
from src.ml.directed.nb import load_train, build, cv_predict

TRUE = "directed"  # The class we're trying to catch.

def two_way(labels):
    """Collapse fixable and unsalvageable into directed, so models line up with the rules."""
    return ["general" if l == "general" else TRUE for l in labels]

def report(truth, pred, labels):
    print(classification_report(truth, pred, labels=labels, zero_division=0))

## 1. Building the models

### Train and test split

The 400 labelled questions were split once, before any labelling, by `sample.py`:
- **Train (300)**: what the models learn from.
- **Test (100)**: locked away. Only the rules and the final winning model get scored on it, once, so it stays an honest check.

In [ ]:
from src.ml.directed.common import TEST_PATH, read_jsonl

pd.DataFrame({
    "train": pd.Series([r["label"] for r in read_jsonl(TRAIN_PATH)]).value_counts(),
    "test": pd.Series([r["label"] for r in read_jsonl(TEST_PATH)]).value_counts(),
}).reindex(LABELS)

### Cross-validation on train

Every model is trained and scored with 5-fold cross-validation on the 300 train questions: train on four fifths, predict the remaining fifth, and repeat so every question gets predicted once. Every model uses the same folds, so the comparison is fair.

In [21]:
texts, labels = load_train()

preds = {
    "dummy": cv_predict(DummyClassifier(strategy="most_frequent"), texts, labels),
    "naive bayes": cv_predict(build(), texts, labels),
}

## 2. Results

Sections 2a to 2c are cross-validation results on train. Section 2d is the only one scored on test.

### 2a. Rules vs dummy vs Naive Bayes: directed vs general

The rules only say yes or no, so to compare them fairly every model's three labels are collapsed into two: directed (fixable or unsalvageable) and general. This answers whether a model beats the rules that are running now.
- **Precision**: of the questions flagged, how many really were directed.
- **Recall**: of the real directed questions, how many got caught.

In [22]:
_, rules_truth, rules_pred = score_rules(TRAIN_PATH)
print("== rules ==")
report(two_way(labels), rules_pred, [TRUE, "general"])
for name, pred in preds.items():
    print(f"== {name} ==")
    report(two_way(labels), two_way(pred), [TRUE, "general"])

== rules ==
              precision    recall  f1-score   support

    directed       0.63      0.20      0.30        97
     general       0.71      0.95      0.81       203

    accuracy                           0.70       300
   macro avg       0.67      0.57      0.56       300
weighted avg       0.69      0.70      0.65       300

== dummy ==
              precision    recall  f1-score   support

    directed       0.00      0.00      0.00        97
     general       0.68      1.00      0.81       203

    accuracy                           0.68       300
   macro avg       0.34      0.50      0.40       300
weighted avg       0.46      0.68      0.55       300

== naive bayes ==
              precision    recall  f1-score   support

    directed       0.83      0.62      0.71        97
     general       0.84      0.94      0.89       203

    accuracy                           0.84       300
   macro avg       0.84      0.78      0.80       300
weighted avg       0.84      0.8

### 2b. Dummy vs Naive Bayes: general vs fixable vs unsalvageable

The real job is picking one of three labels, since fixable questions go to the rewriter and unsalvageable ones get set aside. The rules can't do this, so only models that predict all three labels are scored here.

The main number to watch is **unsalvageable precision** (ADR 012), because wrongly calling a good question unsalvageable is the costly mistake.

In [23]:
for name, pred in preds.items():
    print(f"== {name} ==")
    report(labels, pred, LABELS)

== dummy ==
               precision    recall  f1-score   support

      general       0.68      1.00      0.81       203
      fixable       0.00      0.00      0.00        32
unsalvageable       0.00      0.00      0.00        65

     accuracy                           0.68       300
    macro avg       0.23      0.33      0.27       300
 weighted avg       0.46      0.68      0.55       300

== naive bayes ==
               precision    recall  f1-score   support

      general       0.84      0.94      0.89       203
      fixable       0.38      0.09      0.15        32
unsalvageable       0.69      0.68      0.68        65

     accuracy                           0.79       300
    macro avg       0.63      0.57      0.57       300
 weighted avg       0.76      0.79      0.76       300



### 2c. Naive Bayes mistakes: which label each question ended up in

2b gives a score per label, and this shows where the misses went. Rows are my labels, columns are what Naive Bayes said. The diagonal is correct, and everything else is a mistake.

In [24]:
pd.DataFrame(
    confusion_matrix(labels, preds["naive bayes"], labels=LABELS),
    index=[f"my label: {l}" for l in LABELS],
    columns=[f"nb said: {l}" for l in LABELS],
)

,nb said: general,nb said: fixable,nb said: unsalvageable
my label: general,191,3,9
my label: fixable,18,3,11
my label: unsalvageable,19,2,44


### 2d. Rules on test

The rules aren't trained on anything, so scoring them on the locked test set leaks nothing. The winning model gets scored here once, at the very end.

In [25]:
_, test_truth, test_pred = score_rules()
report(test_truth, test_pred, [TRUE, "general"])

              precision    recall  f1-score   support

    directed       0.73      0.25      0.37        44
     general       0.61      0.93      0.74        56

    accuracy                           0.63       100
   macro avg       0.67      0.59      0.56       100
weighted avg       0.67      0.63      0.58       100

